# Training `laya-vision` on ModernVBERT (Kaggle GPU T4 ×2 DDP)

[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Encoder](https://img.shields.io/badge/%F0%9F%A4%97%20Encoder-ModernVBERT%2Fmodernvbert-orange)](https://huggingface.co/ModernVBERT/modernvbert)

This notebook trains **`laya-vision`**: [ModernVBERT](https://huggingface.co/ModernVBERT/modernvbert) (Ettin-150M text + SigLIP2-base-16-512 + a pixel-shuffle connector) as the encoder, with Laya's `DecisionModel` head on top, so typed questions (`choice` / `score` / `noul`) can be asked about **images** in one forward pass.

It runs the RLCD loop from the typed-decisions notebook — GRPO-style noisy logits, `proper_reward(w_sph=0.75, w_rps=1.0)`, soft-CE guidance, two-LR AdamW — with the SigLIP tower frozen and the head optionally warm-started from `laya-multilingual`.

**This checkpoint is not published.** The code lives on the `sb/vision` branch, not on PyPI, so cell 2 clones the repo.

---

### ⚠️ Kaggle notebook settings
In the right-hand sidebar under **Notebook options**:
* **Accelerator:** `GPU T4 x2`
* **Internet:** `On`
* **Output:** checkpoints are written to `/kaggle/working/laya-vision`

### Order of work
Sections 3-6 are a **smoke test** (a few minutes) that proves the pipeline end to end. Only then does section 7 start the real run. Skipping the smoke test risks discovering a data or DDP problem hours in.

## 1. Environment & dual-T4 check

In [ ]:
!nvidia-smi
import os, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Right sidebar -> Notebook options -> Accelerator -> GPU T4 x2."
)

# transformers probes for TensorFlow at import; TF's abseil runtime can deadlock model
# construction. Laya is torch-only.
os.environ.update(USE_TF="0", USE_TORCH="1", TOKENIZERS_PARALLELISM="false")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")

## 2. Install

`laya-vision` needs **transformers >= 5.3** (the first release with `ModernVBertModel`) and pillow — the `laya[vision]` extra. Kaggle images ship transformers 4.x, so this upgrades it.

Set `REPO_URL` to your fork if you are not using the branch below.

In [ ]:
REPO_URL = "https://github.com/sebastianberns/laya.git"   # <- your fork, if different
BRANCH   = "sb/vision"
WORKDIR  = "/kaggle/working/laya"

import os, shutil
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)          # a clean clone each session avoids a stale checkout
!git clone -q -b {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1

!pip install -q -e ".[vision]" "datasets>=3.0.0"

### ⚠️ Restart the session now

`pip install -e .` points at the working tree, but a kernel that already imported `transformers` or `laya` keeps the **old** modules for its whole life — which silently makes later cells measure or run stale code.

**Run ▸ Restart session**, then continue from the next cell. (`!python` cells are separate processes and would pick up the new code either way; in-kernel cells would not.)

In [ ]:
import os
os.environ.update(USE_TF="0", USE_TORCH="1", TOKENIZERS_PARALLELISM="false")
%cd /kaggle/working/laya

import laya, transformers, torch
print("laya         :", laya.__version__, "|", laya.__file__)
print("transformers :", transformers.__version__, "(needs >= 5.3)")
print("torch        :", torch.__version__)
from transformers import ModernVBertConfig      # fails loudly if transformers is too old
print("ModernVBERT support: OK")

## 3. Smoke test — single GPU

`--smoke` caps every task at 12 examples and stops after 3 optimiser steps, so this checks the whole path (data → images → training → calibration → checkpoint) in minutes.

`rvl_cdip` and `koniq` are left out here: KonIQ alone is a 6.3 GB download. Expect `warm-started 35 head tensors`, three loss lines, fitted temperatures, and a saved checkpoint.

In [ ]:
cmd = ("python research/scripts/train_vision.py --out /kaggle/working/lv_smoke"
       " --smoke --tasks cifar100,vqav2_yesno,typed --init-head multilingual")
print(cmd)
!{cmd}

## 4. Smoke test — 2×T4 DDP

The part worth watching: NCCL init, `find_unused_parameters` with the frozen SigLIP tower, and the rank-0-only save. If it hangs at startup rather than erroring, re-run with `--nproc_per_node=1` to confirm it is the distributed path.

In [ ]:
cmd = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
       " --out /kaggle/working/lv_ddp --smoke --tasks cifar100,typed --init-head random")
print(cmd)
!{cmd}

## 5. Does the checkpoint load, and does routing send images to it?

The answers are meaningless after 3 steps — this checks the plumbing: the checkpoint loads through `Agent`, and `Router` sends an image request to `vision`.

In [ ]:
import laya
from PIL import Image

img = Image.new("RGB", (640, 480), "sienna")
Q = {
    "kind":    {"type": "choice", "instructions": "What is shown?",
                "criteria": ["parcel", "invoice", "person", "vehicle"]},
    "damaged": {"type": "noul", "instructions": "Is the parcel damaged?"},
    "quality": {"type": "score", "instructions": "Photo quality",
                "criteria": ["bad", "poor", "ok", "good", "excellent"]},
}

r = laya.Router(models={"vision": "/kaggle/working/lv_smoke"}, device="cuda")
out = r.predict({"note": "customer photo"}, Q, images=[img])
print("routing:", out["routing"]["reason"])
for qid, ans in out["answers"].items():
    print(" ", qid, ans)

# a text-only checkpoint must refuse images rather than ignore them
try:
    laya.Agent("convaiinnovations/laya", device="cuda").predict("hello", Q, images=[img])
    print("\nBUG: a text checkpoint accepted images")
except ValueError as e:
    print("\ntext checkpoint correctly refuses images:", str(e)[:80], "...")

## 6. What does an image actually cost?

Preprocessing is CPU work and the tower is GPU work, so measure them apart. If `image_block` is anywhere near 100 ms, the size cap is not in effect — check that the restart in section 2 actually happened.

Reference from a previous T4 run: preprocess ~25 ms, tower ~62 ms, `predict` with 1 image ~75 ms against ~27 ms text-only.

In [ ]:
import time, numpy as np, torch, laya
from PIL import Image
from laya.vision import image_block

a = laya.Agent("/kaggle/working/lv_smoke", device="cuda")
img = Image.new("RGB", (800, 600), (120, 90, 60))
Q1 = {"kind": {"type": "choice", "instructions": "What is shown?",
               "criteria": ["parcel", "invoice", "person", "vehicle"]}}

def p50(f, n=7):
    f(); torch.cuda.synchronize(); ts = []
    for _ in range(n):
        t = time.perf_counter(); f(); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    return 1000 * float(np.median(ts))

prefix, pv, pam = image_block(a.processor, [img], 1)
print("image block: %d tokens per image" % len(prefix))
print("image_block (CPU preprocess) %6.1f ms" % p50(lambda: image_block(a.processor, [img], 1)))
with torch.no_grad():
    pv_d = pv.to(a.device)
    print("vision tower + connector     %6.1f ms" % p50(lambda: a.model.encoder.get_image_features(pixel_values=pv_d)))
print("predict, 1 image             %6.1f ms" % p50(lambda: a.predict({"note": "x"}, Q1, images=[img])))
print("predict, text only           %6.1f ms" % p50(lambda: a.predict({"note": "x"}, Q1)))

# the tower runs once per call, so the image amortises across questions
for n in (1, 5, 10):
    qs = {"q%d" % i: Q1["kind"] for i in range(n)}
    ms = p50(lambda: a.predict({"note": "x"}, qs, images=[img]))
    print("  %2d questions, 1 image: %6.1f ms total, %5.1f ms per question" % (n, ms, ms / n))

del a
torch.cuda.empty_cache()

## 7. The real training run

Roughly 30k examples across the chosen tasks, 3 epochs on both T4s.

**Prepare the data first** (next cell). Building a split streams images from the Hub; inside `torchrun` every rank would build the whole thing at once — duplicated downloads, saturated CPU, idle GPUs. `--prepare-only` builds it once in a single process and caches it as one `.pt` file per task and split, holding the encoded image bytes with each question and target. Training then just loads them.

### Preparing without spending GPU time

`--prepare-only` touches no model weights, so it is a pure CPU job and does not need this GPU session:

1. Run the prepare cell in a **CPU-only** Kaggle session (or on any machine) with `--data-cache /kaggle/working/vision_data`. On Kaggle this saves your GPU quota — CPU sessions get the same 4 vCPUs, but do not bill against the ~30 GPU-hours a week.
2. **Save Version** on that notebook, so its output becomes a dataset.
3. In this GPU notebook, **Add Data ▸ Your Datasets**, then point `DATA_CACHE` at `/kaggle/input/<dataset-name>/vision_data`. The cache is read-only there, which is fine — nothing is written when every split is already cached.

The cache is plain `torch.save`d Python objects (bytes and numbers, no tensors or CUDA state), so it moves between machines freely. Cache files are keyed by **task, split, size and seed**: keep `--tasks`, `--n-per-task` and `--seed` identical between preparing and training, or it will simply rebuild.

Rough sizes, measured: CIFAR-100 ~2 KB per example, RVL-CDIP ~22 KB, VQAv2 ~90 KB, KonIQ ~200 KB — about 0.9 GB for the four tasks below, ~2.3 GB with `koniq`.

**The two practical limits for the training run:**
* **Disk.** KonIQ-10k is a 6.3 GB download plus ~2 GB extracted, against Kaggle's ~20 GB working quota. Leave `koniq` out of `TASKS` if space is tight.
* **Time.** Kaggle sessions are capped at 9-12 h. The script writes `checkpoint_latest/` after every epoch, so a timeout loses at most the epoch in progress.

If the GPUs sit idle *during training*, image preprocessing in the DataLoader is the bottleneck: raise `--workers`. If a T4 runs out of memory, halve `--micro-batch` and double `--grad-accum` to keep the effective batch at 64.

In [ ]:
# Build and cache every split once, in one process. No model weights are downloaded, so this
# also runs in a CPU-only session. Progress prints every 1000 examples for the streamed tasks.
TASKS      = "cifar100,rvl_cdip,vqav2_yesno,typed"   # add koniq if you have the disk for it
DATA_CACHE = "/kaggle/working/vision_data"           # or /kaggle/input/<dataset>/vision_data

cmd = ("python research/scripts/train_vision.py --out /kaggle/working/laya-vision"
       f" --tasks {TASKS} --data-cache {DATA_CACHE} --prepare-only")
print(cmd)
!{cmd}

!du -sh {DATA_CACHE} 2>/dev/null; ls -la {DATA_CACHE} 2>/dev/null

In [ ]:
# TASKS / DATA_CACHE come from the prepare cell above (or point DATA_CACHE at an input dataset
# you prepared elsewhere). The data is already cached, so training starts almost immediately.
cmd = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
       f" --out /kaggle/working/laya-vision --tasks {TASKS} --data-cache {DATA_CACHE}"
       " --init-head multilingual --epochs 3 --workers 4")
print(cmd)
!{cmd}

## 8. Benchmark

Held-out test splits for every trained task, plus `pets` (20 breeds, never trained on — the zero-shot probe) and `typed` (the text-regression check against the published 0.766).

Baselines: SigLIP2 zero-shot — the "no decision head" reference — majority class, and random. SigLIP2 is strong on object recognition (~0.94 on CIFAR-100 and Pets in a smoke run) and at chance on VQAv2 yes/no, which similarity scoring cannot answer.

Results are written as measured. The success criteria at the end are checks, not targets to tune — write up any shortfall in `BENCHMARKS.md`'s honest-limits style.

In [ ]:
cmd = ("python research/scripts/bench_vision.py --model vision=/kaggle/working/laya-vision"
       " --n 500 --out /kaggle/working/vision_benchmark.json")
print(cmd)
!{cmd}

In [ ]:
# A head trained from scratch, to isolate what the multilingual warm-start is worth.
# Reuses the same cached data. Skip if you are short on session time.
train = ("torchrun --standalone --nproc_per_node=2 research/scripts/train_vision.py"
         f" --out /kaggle/working/laya-vision-rand --tasks {TASKS} --data-cache {DATA_CACHE}"
         " --init-head random --epochs 3 --workers 4")
bench = ("python research/scripts/bench_vision.py"
         " --model warm=/kaggle/working/laya-vision"
         " --model random=/kaggle/working/laya-vision-rand"
         " --n 500 --out /kaggle/working/vision_benchmark.json")
!{train}
!{bench}

In [ ]:
import json
report = json.load(open("/kaggle/working/vision_benchmark.json"))
for task, r in report["tasks"].items():
    line = "%-12s (%s)" % (task, r["type"])
    for name in report["models"]:
        if name in r:
            line += "  %s acc %.3f ece %.3f" % (name, r[name]["accuracy"], r[name]["after_temp"]["ece"])
    if "siglip2_zero_shot" in r:
        line += "  | siglip2 %.3f" % r["siglip2_zero_shot"]["accuracy"]
    print(line)
print()
print(json.dumps(report["success_criteria"], indent=2))

## 9. (Optional) publish to the Hub

Off by default: publishing is public and hard to undo. Set `PUBLISH = True` deliberately, and only once the benchmark numbers justify it.

Needs a **write** token in Kaggle's **Add-ons ▸ Secrets** as `HF_TOKEN`.

In [ ]:
PUBLISH   = False                                  # <- set True to actually upload
REPO_ID   = "sebastianberns/laya-vision"
LOCAL_DIR = "/kaggle/working/laya-vision"

if not PUBLISH:
    print("PUBLISH is False - nothing uploaded. Review the benchmark first.")
else:
    from huggingface_hub import HfApi
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        raise RuntimeError("No HF_TOKEN found. Add-ons -> Secrets -> HF_TOKEN (a write token).")

    api = HfApi(token=token)
    api.create_repo(REPO_ID, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=LOCAL_DIR,
        repo_id=REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint_latest/*"],    # the rolling checkpoint is not a release
    )
    print("published to https://huggingface.co/%s" % REPO_ID)
    print("Remember: laya-vision is English-first (ModernVBERT's Ettin text side). "
          "Do not claim multilingual image+text decisions on the model card.")